In [ ]:
import numpy as np
import tensorflow as tf
import keras
import keras_tuner as kt
from keras_tuner import (HyperModel,  RandomSearch,  BayesianOptimization)
from tensorflow.keras.layers import (Input, Dense, Dropout, layers,Flatten, TimeDistributed, Bidirectional, LSTM, GRU, BatchNormalization, LayerNormalization,  MultiHeadAttention )
from tensorflow.keras.models import Model,Input
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler, ReduceLROnPlateau
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import RMSprop, Adam
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report )

In [ ]:
# Parameters
use_existing_model = False
ALL_MOTION = [1,2,3,4,5,6]
N_MOTION = len(ALL_MOTION)

 Copy of data_loading_preprocessing.ipynb
Run the script first to load the samples before executing other functions or scripts

In [ ]:
# Learning Rate Scheduler
def scheduler(epoch, lr):
    if epoch < 10:
        return lr
    else:
        return lr * tf.math.exp(-0.1)

lr_scheduler = LearningRateScheduler(scheduler)

# Define the EarlyStopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=10, verbose=1, restore_best_weights=True)

In [ ]:
# Define the HyperModel
class MyHyperModel(HyperModel):
    def __init__(self, input_shape, n_class):
        self.input_shape = input_shape
        self.n_class = n_class

    def build(self, hp):
        model_input = Input(shape=self.input_shape, dtype='float32', name='name_model_input')

        # Feature extraction part
        x = layers.TimeDistributed(layers.Conv2D(
            filters=hp.Int('conv_filters', min_value=16, max_value=64, step=16),
            kernel_size=(5, 5),
            activation='relu',
            data_format='channels_last'
        ))(model_input)
        x = layers.TimeDistributed(layers.MaxPooling2D(pool_size=(2, 2)))(x)
        x = layers.TimeDistributed(layers.Flatten())(x)
        x = layers.TimeDistributed(layers.Dense(
            units=hp.Int('dense_units', min_value=128, max_value=512, step=128),
            activation='gelu'
        ))(x)
        x = layers.Bidirectional(layers.GRU(
            units=hp.Int('gru_units', min_value=32, max_value=128, step=32),
            return_sequences=True
        ))(x)
        x = layers.Dropout(hp.Float('dropout_rate', min_value=0.1, max_value=0.5, step=0.1))(x)
        x = layers.LSTM(
            units=hp.Int('lstm_units', min_value=32, max_value=128, step=32),
            return_sequences=True
        )(x)
        x = layers.Dropout(hp.Float('dropout_rate', min_value=0.1, max_value=0.5, step=0.1))(x)

        # Output part
        x = layers.Flatten()(x)
        model_output = layers.Dense(self.n_class, activation='softmax', name='name_model_output')(x)

        # Model compiling
        model = Model(inputs=model_input, outputs=model_output)
        loss_function = CategoricalCrossentropy(label_smoothing=0.1)
        model.compile(optimizer=RMSprop(learning_rate=0.001),
                      loss=loss_function,
                      metrics=['accuracy'])
        return model

# Instantiate the HyperModel
input_shape = (36, 20, 20, 1)
n_class = N_MOTION
hypermodel = MyHyperModel(input_shape=input_shape, n_class=n_class)

# Instantiate the tuner
tuner = RandomSearch(
    hypermodel,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='my_dir',
    project_name='nas_project'
)


# Run the search for training samples
tuner.search(training_generator,
             epochs=10,
             validation_data=validation_generator)

# Get the best model
best_model = tuner.get_best_models(num_models=1)[0]

# Evaluate the best model
loss, acc = best_model.evaluate(test_generator)
print(f"Test Loss: {loss}, Test Accuracy: {acc}")

In [ ]:
from kerastuner.tuners import BayesianOptimization
# Instantiate the tuner
tuner_1 = BayesianOptimization(
    hypermodel,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='my_dir',
    project_name='nas_bayeproject'
)


# Run the search
tuner_1.search(training_generator,
             epochs=10,
             validation_data=validation_generator)

# Get the best model
best_model_1 = tuner_1.get_best_models(num_models=1)[0]

# Evaluate the best model
loss_1, acc_1 = best_model_1.evaluate(test_generator)
print(f"Test Loss: {loss}, Test Accuracy: {acc}")

In [ ]:
# Evaluate the best model
loss_1, acc_1 = best_model_1.evaluate(test_generator)
print(f"Test Loss: {loss_1}, Test Accuracy: {acc_1}")

In [ ]:
best_model_1.summary()

In [ ]:
import time
# Measure Latency and Throughput for multiple test datasets
def measure_metrics(model, test_datasets):
    results = {}
    for i, test_data in enumerate(test_datasets):
        start_time = time.time()
        predictions = model.predict(test_data)
        end_time = time.time()
        latency = (end_time - start_time) / len(test_data)
        total_time = end_time - start_time
        throughput = len(test_data) / total_time
        results[f'test_{i}'] = {'latency': latency, 'throughput': throughput}
    return results

# List of test datasets
test_datasets = [test_generator, test_1_generator, test_2_generator]

# Measure metrics for all test datasets
metrics = measure_metrics(best_model, test_datasets)

# Print the results
for test_name, metric in metrics.items():
    print(f"{test_name} - Inference Latency: {metric['latency']} seconds per sample, Throughput: {metric['throughput']} samples per second")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Function to evaluate model performance on multiple datasets
def evaluate_model(model, data_generators, loss_function):
    results = {}
    for i, data_generator in enumerate(data_generators):
        y_true = []
        y_pred = []
        total_loss = 0.0
        num_batches = 0
        
        start_time = time.time()
        
        for data, label in data_generator:
            logits = model(data, training=False)
            predictions = np.argmax(logits, axis=1)
            y_true.extend(np.argmax(label, axis=1))
            y_pred.extend(predictions)
            
            # Compute loss
            loss = loss_function(label, logits)
            total_loss += loss.numpy()
            num_batches += 1
        
        end_time = time.time()
        execution_time = end_time - start_time
        
        # Average loss over all batches
        avg_loss = total_loss / num_batches
        
        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, average='weighted')
        recall = recall_score(y_true, y_pred, average='weighted')
        f1 = f1_score(y_true, y_pred, average='weighted')
        conf_matrix = confusion_matrix(y_true, y_pred)
        
        # Normalize the confusion matrix to get percentages
        conf_matrix_percent = conf_matrix.astype('float') / conf_matrix.sum(axis=1)[:, np.newaxis] * 100
        
        class_report = classification_report(y_true, y_pred)
        
        results[f'test_{i}'] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'loss': avg_loss,
            'execution_time': execution_time,
            'class_report': class_report
        }
    return results

# Example loss function (assuming TensorFlow/Keras)
import tensorflow as tf
loss_function = tf.keras.losses.CategoricalCrossentropy(from_logits=True)

# List of test datasets
test_datasets = [test_generator, test_1_generator, test_2_generator]

# Evaluate model on all test datasets
metrics = evaluate_model(best_model, test_datasets, loss_function)

# Print the results
for test_name, metric in metrics.items():
    print(f"{test_name} - Accuracy: {metric['accuracy']}, Precision: {metric['precision']}, Recall: {metric['recall']}, F1 Score: {metric['f1']}, Loss: {metric['loss']}, Execution Time: {metric['execution_time']} seconds")
    print(f"Classification Report:\n{metric['class_report']}")

In [ ]:
# Transformer model
# MultiHeadAttention Layer
# Create positional encoding
def positional_encoding(seq_len, d_model):
    angles = 1 / np.power(10000., (2 * (np.arange(d_model)[np.newaxis, :] // 2)) / np.float32(d_model))
    positional_encoding = np.zeros((seq_len, d_model))
    positional_encoding[:, 0::2] = np.sin(np.dot(np.arange(seq_len)[:, np.newaxis], angles[:, 0::2]))
    positional_encoding[:, 1::2] = np.cos(np.dot(np.arange(seq_len)[:, np.newaxis], angles[:, 1::2]))
    positional_encoding = positional_encoding[np.newaxis, ...]
    return tf.cast(positional_encoding, dtype=tf.float32)

#Create MultiHeadAttention Encoder
class ScaledDotProductAttention(Layer):
    def __init__(self, dropout_ratio=0.1, **kwargs):
        super(ScaledDotProductAttention, self).__init__(**kwargs)
        self.dropout = Dropout(dropout_ratio)

    def call(self, q, k, v, mask=None, training=False):
        matmul_qk = tf.matmul(q, k, transpose_b=True)
        dk = tf.cast(tf.shape(k)[-1], tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

        if mask is not None:
            scaled_attention_logits += (mask * -1e9)

        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        attention_weights = self.dropout(attention_weights, training=training)
        output = tf.matmul(attention_weights, v)
        return output
    
class MultiHeadAttention(Layer):
    def __init__(self, d_model, num_heads, dropout_ratio=0.1, **kwargs):
        super(MultiHeadAttention, self).__init__(**kwargs)
        self.num_heads = num_heads
        self.d_model = d_model
        assert d_model % self.num_heads == 0  # d_model must be divisible by num_heads

        self.depth = d_model // self.num_heads
        self.dropout_ratio = dropout_ratio

        self.wq = Dense(d_model)
        self.wk = Dense(d_model)
        self.wv = Dense(d_model)

        self.dropout = Dropout(dropout_ratio)
        self.dense = Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, q, k, v, mask=None, training=False):
        batch_size = tf.shape(q)[0]

        q = self.wq(q)  # (batch_size, seq_len, d_model)
        k = self.wk(k)  # (batch_size, seq_len, d_model)
        v = self.wv(v)  # (batch_size, seq_len, d_model)

        q = self.split_heads(q, batch_size)  # (batch_size, num_heads, seq_len_q, depth)
        k = self.split_heads(k, batch_size)  # (batch_size, num_heads, seq_len_k, depth)
        v = self.split_heads(v, batch_size)  # (batch_size, num_heads, seq_len_v, depth)

        scaled_attention = ScaledDotProductAttention(self.dropout_ratio)(q, k, v, mask, training)
        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])  # (batch_size, seq_len_q, num_heads, depth)
        concat_attention = tf.reshape(scaled_attention, (batch_size, -1, self.d_model))  # (batch_size, seq_len_q, d_model)
        output = self.dense(concat_attention)  # (batch_size, seq_len_q, d_model)
        return output


class TransformerMultiHeadAttentionEncoderLayer(Layer):
    def __init__(self, ff_dim, dropout_ratio, **kwargs):
        super(TransformerMultiHeadAttentionEncoderLayer, self).__init__(**kwargs)
        self.ff_dim = ff_dim
        self.dropout_ratio = dropout_ratio

    def build(self, input_shape):
        self.attention = MultiHeadAttention(d_model=input_shape[-1], num_heads=4, dropout_ratio=self.dropout_ratio)
        self.layer_norm1 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(self.dropout_ratio)
        self.dense1 = Dense(self.ff_dim, activation='gelu', kernel_regularizer=l2(0.01))
        self.dense2 = Dense(input_shape[-1], kernel_regularizer=l2(0.01))
        self.layer_norm2 = LayerNormalization(epsilon=1e-6)
        self.dropout2 = Dropout(self.dropout_ratio)
        super(TransformerMultiHeadAttentionEncoderLayer, self).build(input_shape)

    def call(self, inputs, training=False):
        seq_len, d_model = inputs.shape[1], inputs.shape[2]
        pos_encoding = positional_encoding(seq_len, d_model)
        x = inputs + pos_encoding

        attn_output = self.attention(x, x, x)  # Q, K, V are the same for self-attention
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layer_norm1(inputs + attn_output)

        ffn_output = self.dense1(out1)
        ffn_output = self.dense2(ffn_output)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layer_norm2(out1 + ffn_output)
    
    def get_config(self):
        config = super(TransformerMultiHeadAttentionEncoderLayer, self).get_config()
        config.update({
            'ff_dim': self.ff_dim,
            'dropout_ratio': self.dropout_ratio
        })
        return config

In [ ]:
# Define the HyperModel class
class TransformerHyperModel(kt.HyperModel):
    def build(self, hp):
        input_shape = (36, 20, 20, 1)
        n_class = N_MOTION
        
        model_input = Input(shape=input_shape, dtype='float32', name='name_model_input')
        
        # Feature extraction part
        x = TimeDistributed(Flatten())(model_input)
        x = TimeDistributed(Dense(hp.Int('dense_units', min_value=64, max_value=256, step=64), activation='gelu'))(x)
        x = TimeDistributed(Dropout(hp.Float('dropout_rate', min_value=0.1, max_value=0.5, step=0.1)))(x)
        
        # Transformer encoder stack
        for i in range(hp.Int('num_layers', 1, 4)):
            x = TransformerMultiHeadAttentionEncoderLayer(
                ff_dim=hp.Int(f'ff_dim_{i}', min_value=128, max_value=512, step=128),
                dropout_ratio=hp.Float(f'dropout_ratio_{i}', min_value=0.1, max_value=0.5, step=0.1)
            )(x)
        
        # Output part
        x = Flatten()(x)
        model_output = Dense(n_class, activation='softmax', name='name_model_output')(x)
        
        # Model compiling
        model = Model(inputs=model_input, outputs=model_output)
        # Define the loss function with label smoothing
        loss_function = CategoricalCrossentropy(label_smoothing=0.1)
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')),
                      loss=loss_function,
                      metrics=['accuracy'])
        return model

# Instantiate the HyperModel
hypermodel_Transformer = TransformerHyperModel()


In [ ]:
from kerastuner.tuners import BayesianOptimization
# Instantiate the tuner
tuner_2 = BayesianOptimization(
    hypermodel_Transformer,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='my_dir',
    project_name='trans_NAS'
)

# Define the search space
tuner_2.search_space_summary()


# Run the search
tuner_2.search(training_generator,
             epochs=10,
             validation_data=validation_generator)

# Get the best model
best_model_2 = tuner_2.get_best_models(num_models=1)[0]

# Evaluate the best model
loss_2, acc_2 = best_model_2.evaluate(test_generator)
print(f"Test Loss: {loss_2}, Test Accuracy: {acc_2}")

In [ ]:
best_model_2.summary()

In [ ]:
# Evaluate model on all test datasets
metrics = evaluate_model(best_model_2, test_datasets, loss_function)

# Print the results
for test_name, metric in metrics.items():
    print(f"{test_name} - Accuracy: {metric['accuracy']}, Precision: {metric['precision']}, Recall: {metric['recall']}, F1 Score: {metric['f1']}, Loss: {metric['loss']}, Execution Time: {metric['execution_time']} seconds")
    print(f"Confusion Matrix (Counts):\n{metric['conf_matrix']}")
    print(f"Confusion Matrix (Percentages):\n{metric['conf_matrix_percent']}")
    print(f"Classification Report:\n{metric['class_report']}")

In [ ]:
# List of test datasets
test_datasets = [test_generator, test_1_generator, test_2_generator]

# Measure metrics for all test datasets
metrics = measure_metrics(best_model_4, test_datasets)

# Print the results
for test_name, metric in metrics.items():
    print(f"{test_name} - Inference Latency: {metric['latency']} seconds per sample, Throughput: {metric['throughput']} samples per second")